# Part 5 — Semantic Mapping

This notebook demonstrates the correct functioning of the semantic mapping pipeline (EP-06, EP-07, EP-08):

| File | Responsibility |
|------|----------------|
| `src/mapping/rag_mapper.py` | RAG-augmented mapping with retrieval + LLM |
| `src/mapping/validator.py` | Two-phase Actor-Critic validation |
| `src/mapping/hitl.py` | Human-in-the-Loop breakpoint for LangGraph |

**Architecture overview:**
```
EnrichedTableSchema + Entity list
    │
    ▼
build_retrieval_query() + retrieve_top_entities()  [BGE-M3 embeddings]
    │  → top-K relevant entities for this table
    │
    ▼
propose_mapping()  [LLM with MAPPING_SYSTEM/MAPPING_USER + few-shot]
    │  → MappingProposal(table_name, mapped_concept, confidence, ...)
    │
    ▼
validate_schema() + critic_review()  [Pydantic + LLM CRITIC]
    │  → CriticDecision(approved, critique)
    │
    ▼
should_interrupt()  [confidence < threshold?]
    │
    ▼
hitl_node()  [LangGraph interrupt() for human review]
    │  → Command(goto=...) after approve/correct/reject
```

> **Note**: LLM and embedding calls require live credentials. Sections that perform actual LLM/embedding
> inference use mocks or pre-computed examples so the notebook runs fully offline.

In [10]:
import sys
from pathlib import Path

project_root = Path().resolve().parents[1]
if str(project_root) not in sys.path:
    sys.path.insert(0, str(project_root))

print(f"Project root: {project_root}")

Project root: /home/marcantoniolopez/Documenti/github/thesis


## 1. `build_retrieval_query()` — Query construction for table embedding

`build_retrieval_query(table)` constructs a dense text query from enriched table metadata.
Priority: `enriched_table_name + table_description` → `table_name + column names`. The richer
the query, the better the embedding similarity with business concepts.

In [11]:
from src.mapping.rag_mapper import build_retrieval_query
from src.models.schemas import TableSchema, ColumnSchema, EnrichedTableSchema, EnrichedColumn

# Create an enriched table schema
table = TableSchema(
    table_name="TB_CUST",
    schema_name="dbo",
    columns=[
        ColumnSchema(name="CUST_ID", data_type="INT", is_primary_key=True),
        ColumnSchema(name="CUST_NM", data_type="VARCHAR(100)"),
        ColumnSchema(name="CUST_EMAIL", data_type="VARCHAR(255)"),
        ColumnSchema(name="REG_DT", data_type="DATE"),
    ],
    ddl_source="CREATE TABLE dbo.TB_CUST (CUST_ID INT PRIMARY KEY, ...);",
)

enriched = EnrichedTableSchema.from_table_schema(table)
enriched.enriched_table_name = "Customer Master"
enriched.table_description = "Stores customer identity data."
enriched.enriched_columns = [
    EnrichedColumn(original_name="CUST_ID", enriched_name="Customer ID"),
    EnrichedColumn(original_name="CUST_NM", enriched_name="Customer Name"),
    EnrichedColumn(original_name="CUST_EMAIL", enriched_name="Customer Email"),
    EnrichedColumn(original_name="REG_DT", enriched_name="Registration Date"),
]

query = build_retrieval_query(enriched)
print("Retrieval query:")
print(f"  '{query}'")

assert "Customer Master" in query
assert "Customer ID" in query
print("✅ Query uses enriched metadata")

Retrieval query:
  'Customer Master | Stores customer identity data. | Customer ID, Customer Name, Customer Email, Registration Date'
✅ Query uses enriched metadata


## 2. `retrieve_top_entities()` — Vector-based entity retrieval

`retrieve_top_entities(query, entities, embeddings, top_k)` embeds the query and all entities,
computes cosine similarity, and returns the top-K most similar entities.

In [12]:
import numpy as np
from unittest.mock import MagicMock
from src.mapping.rag_mapper import retrieve_top_entities
from src.models.schemas import Entity

# Create sample business entities
entities = [
    Entity(
        name="Customer",
        definition="A person or organization that has registered and purchased products.",
        synonyms=["client", "buyer"],
        provenance_text="A Customer places orders.",
        source_doc="business_glossary.pdf",
    ),
    Entity(
        name="Product",
        definition="A sellable item offered by the company.",
        synonyms=["SKU", "Item"],
        provenance_text="Products belong to categories.",
        source_doc="business_glossary.pdf",
    ),
    Entity(
        name="SalesOrder",
        definition="A transaction representing a customer purchase.",
        synonyms=["Order", "PurchaseOrder"],
        provenance_text="A Customer places a SalesOrder.",
        source_doc="business_glossary.pdf",
    ),
]

# Mock embeddings: assign high similarity to Customer-related queries
def mock_embed_query(text):
    # Query containing "Customer" gets high similarity to Customer entity
    vec = np.zeros(4, dtype=np.float32)
    if "Customer" in text or "customer" in text.lower():
        vec[0] = 1.0  # Customer dimension
    else:
        vec[1] = 1.0  # Other dimension
    return vec.tolist()

def mock_embed_docs(texts):
    vecs = []
    for e in entities:
        v = np.zeros(4, dtype=np.float32)
        if e.name == "Customer":
            v[0] = 1.0
        elif e.name == "Product":
            v[1] = 1.0
        else:
            v[2] = 1.0
        vecs.append(v.tolist())
    return vecs

mock_embeddings = MagicMock()
mock_embeddings.embed_query = mock_embed_query
mock_embeddings.embed_documents = mock_embed_docs

query = "Customer Master identity data"
top_entities = retrieve_top_entities(query, entities, mock_embeddings, top_k=2)

print(f"Top {len(top_entities)} entities for query '{query}':")
for e in top_entities:
    print(f"  · '{e.name}': {e.definition[:60]}...")

assert top_entities[0].name == "Customer"
print("✅ Most similar entity is returned first")

Top 2 entities for query 'Customer Master identity data':
  · 'Customer': A person or organization that has registered and purchased p...
  · 'SalesOrder': A transaction representing a customer purchase....
✅ Most similar entity is returned first


## 3. `propose_mapping()` — LLM-based mapping proposal

`propose_mapping(table, entities, llm, few_shot_examples)` calls the LLM with `MAPPING_SYSTEM`/
`MAPPING_USER` prompts to propose a semantic mapping. Handles retries with reflection prompts.

In [13]:
import json
from unittest.mock import MagicMock
from langchain_core.messages import AIMessage, SystemMessage, HumanMessage
from src.mapping.rag_mapper import propose_mapping
from src.models.schemas import MappingProposal

# Mock LLM returning a valid mapping proposal
mock_llm = MagicMock()
mock_llm.invoke.return_value = AIMessage(content='''
{
  "table_name": "TB_CUST",
  "mapped_concept": "Customer",
  "confidence": 0.95,
  "reasoning": "Table contains customer identity columns (ID, Name, Email).",
  "alternative_concepts": ["Client"]
}
''')

few_shot_text = "Example mapping: TB_PROD → Product (confidence: 0.98)"

proposal = propose_mapping(table, entities[:2], mock_llm, few_shot_examples=few_shot_text)

print("Mapping Proposal:")
print(f"  table_name:      '{proposal.table_name}'")
print(f"  mapped_concept:  '{proposal.mapped_concept}'")
print(f"  confidence:      {proposal.confidence}")
print(f"  reasoning:       '{proposal.reasoning}'")
print(f"  alternatives:     {proposal.alternative_concepts}")

assert proposal.mapped_concept == "Customer"
assert proposal.confidence == 0.95
print("✅ propose_mapping returns valid MappingProposal")

{"ts": "2026-03-12T16:38:29", "logger": "src.mapping.rag_mapper", "level": "INFO", "message": "Proposed mapping: 'TB_CUST' \u2192 'Customer' (confidence=0.95)"}


Mapping Proposal:
  table_name:      'TB_CUST'
  mapped_concept:  'Customer'
  confidence:      0.95
  reasoning:       'Table contains customer identity columns (ID, Name, Email).'
  alternatives:     ['Client']
✅ propose_mapping returns valid MappingProposal


In [14]:
# Graceful error handling: various failure modes
print("Testing graceful error handling:")

# 1. LLM timeout/error → returns null mapping
error_llm = MagicMock()
error_llm.invoke.side_effect = RuntimeError("LLM timeout")
error_proposal = propose_mapping(table, [], error_llm)
assert error_proposal.mapped_concept is None
assert error_proposal.confidence == 0.0
print("  ✅ LLM error → null mapping (mapped_concept=None, confidence=0.0)")

# 2. Invalid JSON → returns null mapping
bad_json_llm = MagicMock()
bad_json_llm.invoke.return_value = AIMessage(content="not json at all")
bad_json_proposal = propose_mapping(table, [], bad_json_llm)
assert bad_json_proposal.mapped_concept is None
print("  ✅ Invalid JSON → null mapping")

# 3. Pydantic validation error → returns null mapping
bad_schema_llm = MagicMock()
bad_schema_llm.invoke.return_value = AIMessage(content='{"table_name": "TB_CUST"}')  # missing required fields
bad_schema_proposal = propose_mapping(table, [], bad_schema_llm)
assert bad_schema_proposal.mapped_concept is None
print("  ✅ Pydantic validation error → null mapping")

# 4. Reflection prompt is prepended on retry
retry_llm = MagicMock()
retry_llm.invoke.return_value = AIMessage(content='{"table_name": "TB_CUST", "mapped_concept": "Client", "confidence": 0.85, "reasoning": "x", "alternative_concepts": []}')
retry_proposal = propose_mapping(table, [], retry_llm, reflection_prompt="Previous mapping was rejected: too low confidence.")
assert retry_proposal.mapped_concept == "Client"
# Check that reflection was included in the prompt
call_args = retry_llm.invoke.call_args
user_msg = call_args[0][0][1]  # Second message is HumanMessage
assert "[REFLECTION CRITIQUE" in user_msg.content
print("  ✅ Reflection prompt prepended on retry")

{"ts": "2026-03-12T16:38:29", "logger": "src.mapping.rag_mapper", "level": "WARNING", "message": "LLM mapping call failed for table 'TB_CUST': LLM timeout \u2014 returning null mapping."}
{"ts": "2026-03-12T16:38:29", "logger": "src.mapping.rag_mapper", "level": "WARNING", "message": "Non-JSON mapping response for 'TB_CUST'"}
{"ts": "2026-03-12T16:38:29", "logger": "src.mapping.rag_mapper", "level": "WARNING", "message": "Pydantic validation error for mapping of 'TB_CUST': 3 validation errors for MappingProposal\nmapped_concept\n  Field required [type=missing, input_value={'table_name': 'TB_CUST'}, input_type=dict]\n    For further information visit https://errors.pydantic.dev/2.12/v/missing\nconfidence\n  Field required [type=missing, input_value={'table_name': 'TB_CUST'}, input_type=dict]\n    For further information visit https://errors.pydantic.dev/2.12/v/missing\nreasoning\n  Field required [type=missing, input_value={'table_name': 'TB_CUST'}, input_type=dict]\n    For further inf

Testing graceful error handling:
  ✅ LLM error → null mapping (mapped_concept=None, confidence=0.0)
  ✅ Invalid JSON → null mapping
  ✅ Pydantic validation error → null mapping
  ✅ Reflection prompt prepended on retry


## 4. `validate_schema()` — Pydantic validation layer

`validate_schema(proposal_dict)` validates raw LLM output as a `MappingProposal`.
Returns `(MappingProposal, None)` on success or `(None, error_string)` on failure.

In [15]:
from src.mapping.validator import validate_schema

# Valid proposal dict
valid_dict = {
    "table_name": "TB_CUST",
    "mapped_concept": "Customer",
    "confidence": 0.95,
    "reasoning": "Matches customer data structure.",
    "alternative_concepts": ["Client"],
}

proposal, error = validate_schema(valid_dict)
assert proposal is not None
assert error is None
assert proposal.table_name == "TB_CUST"
print("✅ Valid proposal → returns (MappingProposal, None)")

# Missing required field → returns error
invalid_dict = {"mapped_concept": "Customer"}  # missing table_name, confidence, reasoning
proposal, error = validate_schema(invalid_dict)
assert proposal is None
assert error is not None
assert "table_name" in error.lower() or "confidence" in error.lower()
print(f"✅ Invalid proposal → returns (None, error_string): '{error[:60]}...'")

# Confidence out of range → returns error
oor_dict = {**valid_dict, "confidence": 1.5}
proposal, error = validate_schema(oor_dict)
assert proposal is None
assert error is not None
print("✅ Confidence out of range → returns error")

# Null mapped_concept is allowed (for failed mappings)
null_dict = {**valid_dict, "mapped_concept": None, "confidence": 0.0}
proposal, error = validate_schema(null_dict)
assert proposal is not None
assert proposal.mapped_concept is None
assert proposal.confidence == 0.0
print("✅ Null mapped_concept allowed (for no-mapping case)")

{"ts": "2026-03-12T16:38:29", "logger": "src.mapping.validator", "level": "WARNING", "message": "MappingProposal schema validation failed: 3 validation errors for MappingProposal\ntable_name\n  Field required [type=missing, input_value={'mapped_concept': 'Customer'}, input_type=dict]\n    For further information visit https://errors.pydantic.dev/2.12/v/missing\nconfidence\n  Field required [type=missing, input_value={'mapped_concept': 'Customer'}, input_type=dict]\n    For further information visit https://errors.pydantic.dev/2.12/v/missing\nreasoning\n  Field required [type=missing, input_value={'mapped_concept': 'Customer'}, input_type=dict]\n    For further information visit https://errors.pydantic.dev/2.12/v/missing"}
{"ts": "2026-03-12T16:38:29", "logger": "src.mapping.validator", "level": "WARNING", "message": "MappingProposal schema validation failed: 1 validation error for MappingProposal\nconfidence\n  Input should be less than or equal to 1 [type=less_than_equal, input_value=

✅ Valid proposal → returns (MappingProposal, None)
✅ Invalid proposal → returns (None, error_string): '3 validation errors for MappingProposal
table_name
  Field r...'
✅ Confidence out of range → returns error
✅ Null mapped_concept allowed (for no-mapping case)


## 5. `critic_review()` — Actor-Critic LLM validation

`critic_review(proposal, table, entities, llm)` calls the LLM with `CRITIC_SYSTEM`/
`CRITIC_USER` to audit a structurally valid proposal. Returns `CriticDecision`.

In [16]:
from src.mapping.validator import critic_review
from src.models.schemas import CriticDecision

# Mock LLM returning approved=True
approve_llm = MagicMock()
approve_llm.invoke.return_value = AIMessage(content='''
{
  "approved": true,
  "critique": null,
  "suggested_correction": null
}
''')

good_proposal = MappingProposal(
    table_name="TB_CUST",
    mapped_concept="Customer",
    confidence=0.95,
    reasoning="Table structure matches Customer concept.",
    alternative_concepts=[],
)

decision = critic_review(good_proposal, table, entities, approve_llm)
assert decision.approved is True
print(f"✅ Critic approved: approved={decision.approved}, critique={decision.critique}")

# Critic rejects with feedback
reject_llm = MagicMock()
reject_llm.invoke.return_value = AIMessage(content='''
{
  "approved": false,
  "critique": "TB_CUST contains product columns, not customer data.",
  "suggested_correction": "Product"
}
''')

decision_reject = critic_review(good_proposal, table, entities, reject_llm)
assert decision_reject.approved is False
assert "product" in decision_reject.critique.lower()
print(f"✅ Critic rejected: approved={decision_reject.approved}, critique='{decision_reject.critique}'")

# Conservative fallback: LLM error → approved=True (no infinite retry loop)
error_llm = MagicMock()
error_llm.invoke.side_effect = RuntimeError("Critic LLM timeout")
fallback_decision = critic_review(good_proposal, table, [], error_llm)
assert fallback_decision.approved is True  # conservative default
print("✅ LLM error → conservative approved=True (no infinite loop)")

{"ts": "2026-03-12T16:38:29", "logger": "src.mapping.validator", "level": "INFO", "message": "Critic for 'TB_CUST' \u2192 approved=True, critique=None"}


{"ts": "2026-03-12T16:38:29", "logger": "src.mapping.validator", "level": "INFO", "message": "Critic for 'TB_CUST' \u2192 approved=False, critique='TB_CUST contains product columns, not customer data.'"}
{"ts": "2026-03-12T16:38:29", "logger": "src.mapping.validator", "level": "WARNING", "message": "Critic LLM call failed for table 'TB_CUST': Critic LLM timeout \u2014 approving by default."}


✅ Critic approved: approved=True, critique=None
✅ Critic rejected: approved=False, critique='TB_CUST contains product columns, not customer data.'
✅ LLM error → conservative approved=True (no infinite loop)


## 6. `build_reflection_prompt()` — Reflection prompt formatter

`build_reflection_prompt(role, output_format, error, original_input)` formats the
universal `REFLECTION_TEMPLATE` for LLM retry calls.

In [17]:
from src.mapping.validator import build_reflection_prompt

reflection = build_reflection_prompt(
    role="data governance expert",
    output_format="JSON mapping proposal",
    error="confidence must be <= 1.0, got 1.5",
    original_input='{"table_name": "TB_CUST", "confidence": 1.5}',
)

print("Reflection prompt (first 200 chars):")
print(f"  {reflection[:200]}...")

assert "data governance expert" in reflection
assert "confidence must be" in reflection
assert "{\"table_name\":" in reflection
print("✅ All placeholders filled in REFLECTION_TEMPLATE")

Reflection prompt (first 200 chars):
  You are a data governance expert.

Your previous attempt to generate a JSON mapping proposal failed with the following error:

<previous_error>
confidence must be <= 1.0, got 1.5
</previous_error>

Th...
✅ All placeholders filled in REFLECTION_TEMPLATE


## 7. `should_interrupt()` — HITL trigger predicate

`should_interrupt(state)` returns `True` when the confidence is below the threshold
or the `hitl_flag` is explicitly set. Used as both a node check and conditional edge.

In [18]:
from src.mapping.hitl import should_interrupt
from src.config.settings import get_settings
from src.models.state import BuilderState

settings = get_settings()
print(f"Confidence threshold: {settings.confidence_threshold}")

# Low confidence → interrupt
low_conf_state: BuilderState = {  # type: ignore
    "hitl_flag": False,
    "mapping_proposal": MappingProposal(
        table_name="TB_CUST",
        mapped_concept="Customer",
        confidence=0.75,  # below threshold
        reasoning="Uncertain match.",
        alternative_concepts=[],
    ),
}
assert should_interrupt(low_conf_state) is True
print("✅ Low confidence (<0.90) → interrupt=True")

# High confidence → no interrupt
high_conf_state: BuilderState = {  # type: ignore
    "hitl_flag": False,
    "mapping_proposal": MappingProposal(
        table_name="TB_CUST",
        mapped_concept="Customer",
        confidence=0.95,  # above threshold
        reasoning="Strong match.",
        alternative_concepts=[],
    ),
}
assert should_interrupt(high_conf_state) is False
print("✅ High confidence (>=0.90) → interrupt=False")

# Explicit flag → interrupt regardless of confidence
flagged_state: BuilderState = {  # type: ignore
    "hitl_flag": True,
    "mapping_proposal": MappingProposal(
        table_name="TB_CUST",
        mapped_concept="Customer",
        confidence=0.99,
        reasoning="Very confident.",
        alternative_concepts=[],
    ),
}
assert should_interrupt(flagged_state) is True
print("✅ hitl_flag=True → interrupt=True (even with high confidence)")

# Exactly at threshold → no interrupt
at_threshold_state: BuilderState = {  # type: ignore
    "hitl_flag": False,
    "mapping_proposal": MappingProposal(
        table_name="TB_CUST",
        mapped_concept="Customer",
        confidence=0.90,  # exactly at threshold
        reasoning="Threshold match.",
        alternative_concepts=[],
    ),
}
assert should_interrupt(at_threshold_state) is False
print("✅ Confidence == threshold (0.90) → interrupt=False")


Confidence threshold: 0.9
✅ Low confidence (<0.90) → interrupt=True
✅ High confidence (>=0.90) → interrupt=False
✅ hitl_flag=True → interrupt=True (even with high confidence)
✅ Confidence == threshold (0.90) → interrupt=False


## 8. `build_interrupt_payload()` — Human review payload

`build_interrupt_payload(proposal, entities)` assembles the structured dict delivered
to the human reviewer via the checkpoint store.

In [19]:
from src.mapping.hitl import build_interrupt_payload

proposal = MappingProposal(
    table_name="TB_CUST",
    mapped_concept="Customer",
    confidence=0.75,
    reasoning="Possible match but low confidence.",
    alternative_concepts=["Client"],
)

payload = build_interrupt_payload(proposal, entities)

print("Interrupt payload keys:")
for key in payload:
    print(f"  · {key}")

print(f"\nPayload values:")
print(f"  table_name:          '{payload['table_name']}'")
print(f"  proposed_concept:    '{payload['proposed_concept']}'")
print(f"  confidence:          {payload['confidence']}")
print(f"  reasoning:           '{payload['reasoning']}'")
print(f"  alternatives:         {payload['alternative_concepts']}")
print(f"  provenance_text:      '{payload['provenance_text'][:80]}...'")

# Proposed concept should NOT be in alternatives
assert "Customer" not in payload["alternative_concepts"]
print("\n✅ Proposed concept excluded from alternatives")

# Alternatives capped at 4
assert len(payload["alternative_concepts"]) <= 4
print("✅ Alternatives capped at 4")

Interrupt payload keys:
  · table_name
  · proposed_concept
  · confidence
  · reasoning
  · alternative_concepts
  · provenance_text

Payload values:
  table_name:          'TB_CUST'
  proposed_concept:    'Customer'
  confidence:          0.75
  reasoning:           'Possible match but low confidence.'
  alternatives:         ['Client', 'Product', 'SalesOrder']
  provenance_text:      'A Customer places orders. | Products belong to categories. | A Customer places a...'

✅ Proposed concept excluded from alternatives
✅ Alternatives capped at 4


## 9. `hitl_node()` — LangGraph interrupt node

`hitl_node(state)` is a LangGraph node that calls `interrupt()` to suspend execution
and wait for human input. Returns a `Command` with routing instructions.

In [20]:
from unittest.mock import patch
from src.mapping.hitl import hitl_node

# High confidence: pass-through (no interrupt)
cmd = hitl_node(high_conf_state)
assert cmd.goto == "Generate_Cypher"
print("✅ High confidence → Command(goto='Generate_Cypher') pass-through")

# Low confidence: interrupt fires, human approves
with patch("src.mapping.hitl.interrupt", return_value={"action": "approve"}):
    cmd_approve = hitl_node(low_conf_state)
assert cmd_approve.goto == "Generate_Cypher"
print("✅ Low confidence + human approve → Command(goto='Generate_Cypher')")

# Human corrects the concept
with patch("src.mapping.hitl.interrupt", return_value={"action": "correct", "mapped_concept": "Client"}):
    cmd_correct = hitl_node(low_conf_state)
assert cmd_correct.goto == "Generate_Cypher"
assert cmd_correct.update["mapping_proposal"].mapped_concept == "Client"
assert cmd_correct.update["mapping_proposal"].confidence == 1.0
print("✅ Human correct → Command with updated proposal (confidence=1.0)")

# Human rejects the mapping
with patch("src.mapping.hitl.interrupt", return_value={"action": "reject"}):
    cmd_reject = hitl_node(low_conf_state)
assert cmd_reject.goto == "End"
assert cmd_reject.update["rejected"] is True
print("✅ Human reject → Command(goto='End', rejected=True)")

# Unknown action → defaults to approve
with patch("src.mapping.hitl.interrupt", return_value={"action": "???"}):
    cmd_unknown = hitl_node(low_conf_state)
assert cmd_unknown.goto == "Generate_Cypher"
print("✅ Unknown action → defaults to approve")

{"ts": "2026-03-12T16:38:29", "logger": "src.mapping.hitl", "level": "INFO", "message": "HITL interrupt fired for table 'TB_CUST' (confidence=0.75)."}
{"ts": "2026-03-12T16:38:29", "logger": "src.mapping.hitl", "level": "INFO", "message": "HITL: human approved mapping 'Customer'."}
{"ts": "2026-03-12T16:38:29", "logger": "src.mapping.hitl", "level": "INFO", "message": "HITL interrupt fired for table 'TB_CUST' (confidence=0.75)."}
{"ts": "2026-03-12T16:38:29", "logger": "src.mapping.hitl", "level": "INFO", "message": "HITL: human corrected 'Customer' \u2192 'Client'."}
{"ts": "2026-03-12T16:38:29", "logger": "src.mapping.hitl", "level": "INFO", "message": "HITL interrupt fired for table 'TB_CUST' (confidence=0.75)."}
{"ts": "2026-03-12T16:38:29", "logger": "src.mapping.hitl", "level": "INFO", "message": "HITL: human rejected mapping for table 'TB_CUST'."}
{"ts": "2026-03-12T16:38:29", "logger": "src.mapping.hitl", "level": "INFO", "message": "HITL interrupt fired for table 'TB_CUST' (co

✅ High confidence → Command(goto='Generate_Cypher') pass-through
✅ Low confidence + human approve → Command(goto='Generate_Cypher')
✅ Human correct → Command with updated proposal (confidence=1.0)
✅ Human reject → Command(goto='End', rejected=True)
✅ Unknown action → defaults to approve


## Summary — Part 5 Architecture

```
┌─────────────────────────────────────────────────────────────────────────────┐
│              EP-06, EP-07, EP-08 — Semantic Mapping Pipeline               │
├──────────────────┬──────────────────┬──────────────────┬──────────────────────┤
│  rag_mapper.py   │  validator.py    │    hitl.py       │                      │
│                  │                  │                  │                      │
│  build_retrieval │  validate_schema │  should_interrupt│                      │
│    _query()      │    (Pydantic)    │    (predicate)   │                      │
│                  │                  │                  │                      │
│  retrieve_top    │  critic_review() │  build_interrupt │                      │
│    _entities()   │    (LLM Actor)   │    _payload()    │                      │
│  [embeddings,    │                  │                  │                      │
│   cosine sim]    │  build_reflection│  hitl_node()     │                      │
│                  │    _prompt()     │  [LangGraph       │                      │
│  propose_mapping │                  │   interrupt()]    │                      │
│    (LLM RAG)     │  Conservative    │  approve/correct/ │                      │
│  graceful []     │    no-merge      │  reject routing  │                      │
└──────────────────┴──────────────────┴──────────────────┴──────────────────────┘
```

**Key design decisions:**
- **Graceful degradation**: every LLM call catches errors and returns safe defaults
- **Conservative defaults**: LLM failures return `confidence=0.0` or `approved=True`
- **Layered validation**: Pydantic (structure) → Actor-Critic (semantics) → Human (uncertain cases)
- **Configurable threshold**: `settings.confidence_threshold` controls HITL sensitivity
- **LangGraph integration**: `Command` objects return both state updates and routing

**Data flow:**
1. `EnrichedTableSchema` + `Entity[]` → RAG retrieval → top-K entities
2. Few-shot examples + table DDL + entities → LLM → `MappingProposal`
3. `MappingProposal` → Pydantic validation → LLM Critic → `CriticDecision`
4. Low confidence → `interrupt()` → human input → `Command(approve/correct/reject)`